# BatchNorm + Dropout CNN
Architecture 2: deeper network with Batch Normalization and Dropout to address the overfitting seen in the baseline.

In [ ]:
pip install torch torchvision torchaudio pandas numpy matplotlib scikit-learn wandb

# Colab & Kaggle Setup
Run this section only in Google Colab. Paste your Kaggle access token below.

In [ ]:
!mkdir -p ~/.kaggle

!echo "PASTE_YOUR_TOKEN_HERE" > ~/.kaggle/access_token

!chmod 600 ~/.kaggle/access_token

!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q challenges-in-representation-learning-facial-expression-recognition-challenge.zip -d data/

# WandB Login

In [ ]:
import wandb
wandb.login()

# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split

import wandb

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Load Data

In [ ]:
DATA_DIR = "data"  # Colab path after Kaggle download
# DATA_DIR = "../data"  # uncomment if running locally

train_df = pd.read_csv(f"{DATA_DIR}/train.csv")

print(train_df.shape)
train_df.head()

# Train / Validation Split

In [ ]:
train_data, val_data = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    stratify=train_df["emotion"]
)

print("Train:", train_data.shape)
print("Validation:", val_data.shape)

# Dataset

In [ ]:
class FERDataset(Dataset):
    def __init__(self, dataframe):
        self.dataframe = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        pixels = np.array(row["pixels"].split(), dtype=np.float32)
        image = pixels.reshape(48, 48)
        image = image / 255.0

        image = torch.tensor(image, dtype=torch.float32).unsqueeze(0)

        label = int(row["emotion"])
        label = torch.tensor(label, dtype=torch.long)

        return image, label

In [ ]:
train_dataset = FERDataset(train_data)
val_dataset = FERDataset(val_data)

image, label = train_dataset[0]
print("Image shape:", image.shape)
print("Label:", label)

# DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)

images, labels = next(iter(train_loader))
print("Batch images shape:", images.shape)
print("Batch labels shape:", labels.shape)

# BatchNorm + Dropout CNN

Changes from baseline:
- 3 conv layers instead of 2 (adding 128 filters in the third layer)
- BatchNorm2d after every conv layer. normalizes activations, stabilizes and speeds up training
- Dropout (0.5) before the final linear layer, randomly disables 50% of neurons during training to prevent memorization
- Wider classifier (256 units instead of 128) to match the increased capacity of the feature extractor

In [ ]:
class BatchNormDropoutCNN(nn.Module):
    def __init__(self, num_classes=7):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 6 * 6, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

# Sanity Checks
Verify forward and backward passes work before full training.

In [ ]:
model = BatchNormDropoutCNN().to(device)

images, labels = next(iter(train_loader))
images = images.to(device)

outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

outputs = model(images)
loss = criterion(outputs, labels)

optimizer.zero_grad()
loss.backward()
optimizer.step()

print("Loss:", loss.item())

# WandB Setup

In [ ]:
run = wandb.init(
    entity="nestor_dzadzamia-free-university-of-tbilisi-",
    project="facial-expression-recognition",
    name="batchnorm-dropout-cnn",
    config={
        "architecture": "BatchNormDropoutCNN",
        "dataset": "FER2013",
        "epochs": 30,
        "batch_size": 64,
        "learning_rate": 1e-3,
        "optimizer": "Adam",
        "num_conv_layers": 3,
        "filters": "32->64->128",
        "fc_units": 256,
        "dropout": 0.5,
        "batch_norm": True,
    }
)

# Training Loop

In [ ]:
NUM_EPOCHS = run.config.epochs

model = BatchNormDropoutCNN().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=run.config.learning_rate)

for epoch in range(NUM_EPOCHS):

    # --- Training phase ---
    model.train()
    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += images.size(0)

    train_loss /= train_total
    train_acc = train_correct / train_total

    # --- Validation phase ---
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            preds = outputs.argmax(dim=1)
            val_correct += (preds == labels).sum().item()
            val_total += images.size(0)

    val_loss /= val_total
    val_acc = val_correct / val_total

    print(f"Epoch {epoch+1}/{NUM_EPOCHS} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_acc": train_acc,
        "val_loss": val_loss,
        "val_acc": val_acc,
    })

In [ ]:
wandb.finish()